# Generative Models: Variational Autoencoders


# Variational autoencoders

In this activity we will build a **variational autoencoder (VAE)** in TensorFlow/Keras. A standard autoencoder maps each input to one point in a latent space. A VAE instead learns a distribution—described here by a mean and log variance—and samples from it during training.

The loss has two terms:

1. a reconstruction loss, which rewards faithful output images; and
2. a Kullback–Leibler (KL) divergence, which regularizes the latent distribution toward a unit Gaussian.

We will first train on MNIST and inspect its two-dimensional latent space. Then you will adapt the same ideas to AT-TPC detector images.

<img src="https://www.jeremyjordan.me/content/images/2018/03/Screen-Shot-2018-03-06-at-3.17.13-PM.png" width="400" />


In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import h5py
from sklearn.cluster import KMeans

layers = tf.keras.layers
tf.keras.utils.set_random_seed(42)


In [ ]:
def load_attpc_data(sim_limit=10_000):
    """Load normalized AT-TPC images without materializing all 50,000 simulations."""
    simulated_origin = (
        "https://github.com/CompPhysics/MachineLearningMSU/raw/master/"
        "Day2_materials/data/simulated-attpc-events.h5"
    )
    real_origin = (
        "https://github.com/CompPhysics/MachineLearningMSU/raw/master/"
        "Day2_materials/data/real-attpc-events.h5"
    )
    simulated_path = tf.keras.utils.get_file(
        "simulated-attpc-data.h5", origin=simulated_origin
    )
    real_path = tf.keras.utils.get_file("real-attpc-data.h5", origin=real_origin)

    with h5py.File(simulated_path, "r") as h5:
        stop = min(sim_limit, len(h5["features"]))
        simulated_features = np.asarray(h5["features"][:stop], dtype=np.float32) / 255.0
        simulated_targets = h5["targets"][:stop]

    with h5py.File(real_path, "r") as h5:
        real_features = np.asarray(h5["features"][:], dtype=np.float32) / 255.0
        real_targets = h5["targets"][:]

    return (real_features, real_targets), (simulated_features, simulated_targets)


def plot_learning_curve(history):
    plt.figure(figsize=(9, 5))
    for key in ("loss", "val_loss", "reconstruction_loss", "kl_loss"):
        if key in history.history:
            plt.plot(history.history[key], label=key)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()


## MNIST data

Load the images, scale their integer pixel values to `[0, 1]`, and flatten each 28 × 28 image into a 784-element vector. Keeping the arrays as `float32` avoids silently doubling their memory use.


In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()


In [ ]:
print('Training Features:\n   Shape: {}\n   Type: {}\n'.format(x_train.shape, x_train.dtype))

In [ ]:
plt.figure(figsize=(10, 10))

for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(x_train[i], cmap=plt.cm.binary)
    
plt.show()

### Scale to `[0, 1]` and flatten the images


In [ ]:
x_train = x_train.astype("float32").reshape(-1, 784) / 255.0
x_test = x_test.astype("float32").reshape(-1, 784) / 255.0


## Build the encoder and decoder

We use a two-dimensional latent space so that we can visualize it. The encoder produces `z_mean` and `z_log_var`; the `Sampling` layer applies the reparameterization trick

$$z = \mu + \exp(\tfrac{1}{2}\log\sigma^2)\,\epsilon, \qquad \epsilon \sim \mathcal{N}(0, I).$$


In [ ]:
latent_dim = 2


class Sampling(layers.Layer):
    """Sample z while keeping the path differentiable."""
    def call(self, inputs):
        z_mean, z_log_var = inputs
        epsilon = tf.random.normal(shape=tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon


encoder_inputs = layers.Input(shape=(784,), name="mnist_image")
x = layers.Dense(256, activation="relu")(encoder_inputs)
x = layers.Dense(128, activation="relu")(x)
z_mean = layers.Dense(latent_dim, name="z_mean")(x)
z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
z = Sampling(name="z")([z_mean, z_log_var])
encoder = tf.keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")

latent_inputs = layers.Input(shape=(latent_dim,), name="latent_sample")
x = layers.Dense(128, activation="relu")(latent_inputs)
x = layers.Dense(256, activation="relu")(x)
decoder_outputs = layers.Dense(784, activation="sigmoid")(x)
decoder = tf.keras.Model(latent_inputs, decoder_outputs, name="decoder")


## Inspect both halves of the model


In [ ]:
encoder.summary()
decoder.summary()


## Define the VAE objective

Keras lets us customize `train_step` while retaining the familiar `compile` and `fit` interface. The reconstruction term below is binary cross-entropy summed over pixels; the KL term encourages a smooth, sampleable latent space.


In [ ]:
class VAE(tf.keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.reconstruction_loss_tracker = tf.keras.metrics.Mean(
            name="reconstruction_loss"
        )
        self.kl_loss_tracker = tf.keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker,
        ]

    def call(self, inputs, training=False):
        _, _, z = self.encoder(inputs, training=training)
        return self.decoder(z, training=training)

    @staticmethod
    def loss_terms(data, reconstruction, z_mean, z_log_var):
        eps = tf.keras.backend.epsilon()
        reconstruction = tf.clip_by_value(reconstruction, eps, 1.0 - eps)
        pixel_bce = -(
            data * tf.math.log(reconstruction)
            + (1.0 - data) * tf.math.log(1.0 - reconstruction)
        )
        reconstruction_loss = tf.reduce_mean(tf.reduce_sum(pixel_bce, axis=1))
        kl_loss = -0.5 * tf.reduce_mean(
            tf.reduce_sum(
                1.0 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=1
            )
        )
        return reconstruction_loss, kl_loss

    def train_step(self, data):
        if isinstance(data, tuple):
            data = data[0]
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data, training=True)
            reconstruction = self.decoder(z, training=True)
            reconstruction_loss, kl_loss = self.loss_terms(
                data, reconstruction, z_mean, z_log_var
            )
            total_loss = reconstruction_loss + kl_loss
        gradients = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_weights))
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {metric.name: metric.result() for metric in self.metrics}

    def test_step(self, data):
        if isinstance(data, tuple):
            data = data[0]
        z_mean, z_log_var, z = self.encoder(data, training=False)
        reconstruction = self.decoder(z, training=False)
        reconstruction_loss, kl_loss = self.loss_terms(
            data, reconstruction, z_mean, z_log_var
        )
        total_loss = reconstruction_loss + kl_loss
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {metric.name: metric.result() for metric in self.metrics}


### Compile the VAE with an explicit, current Adam learning-rate argument


In [ ]:
vae = VAE(encoder, decoder, name="vae")
vae.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3))


### Train the VAE


In [ ]:
history = vae.fit(
    x_train,
    x_train,
    epochs=15,
    batch_size=256,
    shuffle=True,
    validation_data=(x_test, x_test),
)
plot_learning_curve(history)


In [ ]:
# Use the mean for stable, deterministic reconstructions.
z_mean_test, z_log_var_test, z_test = encoder.predict(x_test, verbose=0)
decoded_imgs = decoder.predict(z_mean_test, verbose=0)


In [ ]:
n = 10
plt.figure(figsize=(20, 4))
for i in range(n):
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap="gray")
    ax.axis("off")

    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(decoded_imgs[i].reshape(28, 28), cmap="gray")
    ax.axis("off")
plt.suptitle("Originals (top) and VAE reconstructions (bottom)")
plt.show()


## Inspect the latent space

Because `latent_dim = 2`, we can plot the learned means directly. Labels are used only for coloring this diagnostic plot; the VAE itself trains without labels.


In [ ]:
plt.figure(figsize=(8, 6))
scatter = plt.scatter(z_mean_test[:, 0], z_mean_test[:, 1], c=y_test,
                      s=3, cmap="tab10", alpha=0.6)
plt.colorbar(scatter, ticks=range(10), label="digit")
plt.xlabel("$z_1$")
plt.ylabel("$z_2$")
plt.title("MNIST in the learned latent space")
plt.show()

# Optional: compare an unsupervised clustering with the visible structure.
clust = KMeans(n_clusters=10, n_init=10, random_state=42).fit(z_mean_test)
print("First 20 cluster assignments:", clust.labels_[:20])


In [ ]:
# Decode a regular grid to see whether the latent space is smooth and generative.
grid = np.linspace(-3.0, 3.0, 15)
canvas = np.zeros((28 * len(grid), 28 * len(grid)))
for row, z2 in enumerate(grid[::-1]):
    latent_points = np.column_stack([grid, np.full_like(grid, z2)]).astype("float32")
    generated = decoder.predict(latent_points, verbose=0).reshape(-1, 28, 28)
    for col, digit in enumerate(generated):
        canvas[row * 28:(row + 1) * 28, col * 28:(col + 1) * 28] = digit

plt.figure(figsize=(10, 10))
plt.imshow(canvas, cmap="gray")
plt.axis("off")
plt.title("Samples generated across the VAE latent space")
plt.show()


# AT-TPC extension

Now adapt the VAE to AT-TPC data. The loader reads only 10,000 of the 50,000 simulated images and converts them directly to normalized `float32`, which keeps the Colab RAM footprint manageable and preserves zero-valued pixels.

Questions to consider:

- Will you flatten the 128 × 128 images and use dense layers, or use convolutional layers?
- How large should the latent space be?
- Should you train on simulated images, real images, or both?
- How will you determine whether the latent space represents physically meaningful structure?


In [ ]:
(real_features, _), (sim_features, _) = load_attpc_data(sim_limit=10_000)


Let's confirm the shapes, dtypes, and ranges of the normalized arrays:


In [ ]:
print("Real:", real_features.shape, real_features.dtype,
      (real_features.min(), real_features.max()))
print("Simulated:", sim_features.shape, sim_features.dtype,
      (sim_features.min(), sim_features.max()))


The simulated subset was sliced while reading the HDF5 dataset, before it was copied into RAM. No second 50,000-image array is created.


In [ ]:
print(f"Using {len(sim_features):,} simulated events")


#### Experimental data


In [ ]:
plt.figure(figsize=(10, 10))

for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(real_features[i], cmap='gray')
    
plt.show()

#### Simulated data


In [ ]:
plt.figure(figsize=(10, 10))

for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(sim_features[i], cmap='gray')
    
plt.show()

## Your turn: build an AT-TPC VAE

Start by creating a train/validation split. Then build an encoder that outputs `z_mean`, `z_log_var`, and `z`, plus a decoder whose output shape matches an AT-TPC image. Reuse the VAE loss above.

For a convolutional design, remember that the decoder must reverse the encoder's spatial downsampling. Begin with a small subset and a few epochs, confirm that the loss decreases and reconstructions are sensible, and only then scale up.


In [ ]:
# TODO: build and train your AT-TPC VAE here.
raise NotImplementedError("Design the AT-TPC encoder and decoder")
